# PDA / PBA EDA Notebook

Auto-generated notebook with data quality checks, summaries, time series, and multicollinearity diagnostics.

In [ ]:

import pandas as pd
import numpy as np
import os

print("Working directory:", os.getcwd())
print("Files:", os.listdir("."))


In [ ]:

# ---- read data ----
path = "./pba_model_input_2501.csv"  # adjust if needed
df = pd.read_csv(path)

# ---- basic cleanup ----
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)

df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]
df.head()


## Column Mapping
Edit these once if names differ.

In [ ]:

COL_DATE      = "week_start"
COL_SEGMENT   = "segment"
COL_CHANNEL   = "channel"
COL_CAMPAIGN  = "campaign"
COL_CREATIVE  = "creative_id"
COL_SPEND     = "spend"
COL_IMPR      = "impressions"
COL_CLICKS    = "clicks"


## Data Quality & Sanity Checks

In [ ]:

df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce")

for c in [COL_SPEND, COL_IMPR, COL_CLICKS]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[[COL_DATE, COL_SEGMENT, COL_CHANNEL, COL_SPEND, COL_IMPR, COL_CLICKS]].isna().mean().sort_values(ascending=False)


In [ ]:

KEYS = [COL_DATE, COL_SEGMENT, COL_CHANNEL, COL_CAMPAIGN, COL_CREATIVE]
dupes = df[df.duplicated(KEYS, keep=False)].sort_values(KEYS)
print("Duplicate rows:", len(dupes))
dupes.head(20)


In [ ]:

issues = {
    "negative_spend": df[df[COL_SPEND] < 0],
    "zero_spend": df[df[COL_SPEND] == 0],
    "negative_impr": df[df[COL_IMPR] < 0],
    "negative_clicks": df[df[COL_CLICKS] < 0],
    "clicks_gt_impr": df[df[COL_CLICKS] > df[COL_IMPR]],
}

{k: len(v) for k, v in issues.items()}


## Aggregation & Normalization

In [ ]:

df["cpm"] = np.where(df[COL_IMPR] > 0, (df[COL_SPEND] / df[COL_IMPR]) * 1000, np.nan)
df["cpc"] = np.where(df[COL_CLICKS] > 0, (df[COL_SPEND] / df[COL_CLICKS]), np.nan)

df[[COL_SPEND, COL_IMPR, COL_CLICKS, "cpm", "cpc"]].head()


## Fiscal Year & Summaries

In [ ]:

def fiscal_year(dts, start_month=4):
    return np.where(dts.dt.month >= start_month, dts.dt.year + 1, dts.dt.year)

df["fiscal_year"] = fiscal_year(df[COL_DATE])
df[["fiscal_year", COL_DATE]].head()


In [ ]:

group_cols = ["fiscal_year", COL_SEGMENT, COL_CHANNEL, COL_CAMPAIGN]

fy_summary = (
    df.groupby(group_cols, dropna=False)
      .agg(
          spend_total=(COL_SPEND, "sum"),
          impr_total=(COL_IMPR, "sum"),
          clicks_total=(COL_CLICKS, "sum"),
      )
      .reset_index()
)

fy_summary.head()


## Pivot Tables

In [ ]:

pivot_totals = pd.pivot_table(
    df,
    index=COL_SEGMENT,
    columns=COL_CHANNEL,
    values=COL_SPEND,
    aggfunc="sum",
    fill_value=0
)

pivot_pct = pivot_totals.div(pivot_totals.sum(axis=1).replace(0, np.nan), axis=0)

pivot_totals.head(), pivot_pct.head()


## Time Series

In [ ]:

weekly = (
    df.groupby([COL_DATE, COL_SEGMENT, COL_CHANNEL], dropna=False)[COL_SPEND]
      .sum()
      .reset_index()
      .sort_values(COL_DATE)
)

weekly.head()


## Correlation & VIF

In [ ]:

wide_all = (
    weekly.groupby([COL_DATE, COL_CHANNEL], dropna=False)[COL_SPEND]
          .sum()
          .reset_index()
          .pivot(index=COL_DATE, columns=COL_CHANNEL, values=COL_SPEND)
          .fillna(0)
)

corr = wide_all.corr(method="pearson")
corr.head()


In [ ]:

from statsmodels.stats.outliers_influence import variance_inflation_factor

X = wide_all.loc[:, wide_all.nunique() > 1]
Xv = X.values

vif = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(Xv, i) for i in range(Xv.shape[1])]
}).sort_values("VIF", ascending=False)

vif.head(20)
